# Train on the collected pairs (Google Colab, GPU)
Runtime > Change runtime type > GPU. The shared dataset must be on Drive as `CV_dataset/` with `uav/`, `svi/` and `labels/labels_<name>.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/CV_dataset'
TRAIN_REPO = 'https://github.com/RKsGIS/Assessing-Building-Heat-Resilience-Using-UAV-and-Street-Imagery-with-Coupled-Vision-Transformers.git'
PIPELINE_REPO = 'https://github.com/RKsGIS/UAV-SVI-Annotation-Pipeline.git'

In [ ]:
%cd /content
!git clone -q {TRAIN_REPO} training_repo
!git clone -q -b student {PIPELINE_REPO} pipeline
%cd /content/training_repo
!pip install -q -r requirements.txt

## Prepare the data
`PANO_ONLY = True` uses 360 panoramas only (recommended for a first run); set it to `False` to include ordinary photos too.

In [ ]:
PANO_ONLY = True
flag = '--pano-only' if PANO_ONLY else ''
!python /content/pipeline/model/prepare_training_data.py --data-dir {DATA_DIR} --out-dir /content/training_repo/output {flag}

## Train
Tasks: `structural_openness`, `number_of_floors`, `vegetation`, `material_rooftop`, `material_wall`. Input type `svi` or `uav`. Classes with fewer than 5 samples are dropped by the training script (`--min_samples_per_class`).

In [ ]:
TASK = 'vegetation'
INPUT_TYPE = 'svi'   # 'svi' or 'uav'
!python training/train_classifier_single_input.py --task {TASK} --input_type {INPUT_TYPE} --backbone Tiny --epochs 100 --batch_size 8 --k_folds 5 --test_size 0.15

## Keep the results
Each run writes metrics and checkpoints to `output/training_results/`. Copy them to Drive:

In [ ]:
!cp -r /content/training_repo/output/training_results {DATA_DIR}/